<a href="https://colab.research.google.com/github/ranggahadiwibowo/Indonesia-Healthcare-Infrastructure-Analysis/blob/main/hospital_inequality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# Install DuckDB and Pandas
!pip install duckdb

import duckdb
import pandas as pd
from IPython.display import display

# Data Understanding

### General Overview of the Datasets

Inspect the structure of each dataset to identify column names, data types, and missing values. Based on this information, determine which columns will be used in the following steps. Column names and their order are also adjusted to improve readability.

In [25]:
# Determine hospital dataset file variable
df_h = duckdb.sql("""
SELECT *
FROM '/content/drive/MyDrive/indonesia_hospitals.csv'
""").df()

# Get general info about the dataset
df_h.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3155 entries, 0 to 3154
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  3155 non-null   int64 
 1   nama                3155 non-null   object
 2   propinsi            3155 non-null   object
 3   kab                 3155 non-null   object
 4   alamat              3155 non-null   object
 5   jenis               3155 non-null   object
 6   kelas               3155 non-null   object
 7   status_blu          3155 non-null   object
 8   kepemilikan         3155 non-null   object
 9   total_tempat_tidur  3155 non-null   int64 
 10  total_layanan       3155 non-null   int64 
 11  total_tenaga_kerja  3155 non-null   int64 
dtypes: int64(4), object(8)
memory usage: 295.9+ KB


In [26]:
# Rename field title of the dataset for easier usage
df_h = df_h.rename(columns = {
    'nama': 'hospital_name',
    'propinsi': 'province',
    'kab': 'city',
    'alamat': 'address',
    'jenis': 'type',
    'kelas': 'class',
    'status_blu': 'blu_status',
    'kepemilikan': 'ownership',
    'total_tempat_tidur': 'total_beds',
    'total_layanan': 'total_services',
    'total_layanan': 'total_services',
    'total_tenaga_kerja': 'total_labors'
})

# Re-sort field title of the dataset for easier usage
df_h = df_h[[
    'id', 'province', 'city', 'hospital_name', 'type', 'class',
    'total_beds', 'total_services', 'total_labors', 'ownership',
    'blu_status', 'address'
]]

# Show the result
df_h.head()

,id,province,city,hospital_name,type,class,total_beds,total_services,total_labors,ownership,blu_status,address
0,1110053,Aceh,Kota Lhokseumawe,RS Arun Lhokseumawe,Rumah Sakit Umum,C,218,36,328,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Plaju Komplek Perumahan PT Arun Batuphat T...
1,1106014,Aceh,Aceh Tengah,RS Umum Fandika,Rumah Sakit Umum,D,45,15,45,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Terminal Simpang Wariji Blangkolak 1 Kec. ...
2,1171110,Aceh,Kota Banda Aceh,RS Umum Daerah Meuraxa,Rumah Sakit Umum,B,310,77,487,Pemkot,BLUD,Jl. Soekarno Hatta Km. 2 Desa Mibo Kecamatan B...
3,1171163,Aceh,Kota Banda Aceh,RS Gigi Mulut Universitas Syiah Kuala,Rumah Sakit Khusus Gigi dan Mulut,B,11,24,0,Kementerian Lain,BLU,Jl. Prof A. Madjid Ibrahim I No. 5 Banda Aceh ...
4,1102027,Aceh,Kota Subulussalam,RS Umum Daerah Kota Subulussalam,Rumah Sakit Umum,C,189,34,537,Pemkot,BLUD,Jl. Hamzah Fansyuri (Subulussalam-Rundeng) K...


In [27]:
# Determine population dataset file variable
df_p = duckdb.sql("""
SELECT *
FROM '/content/drive/MyDrive/indonesia_population_2026.csv'
""").df()

# Get general info about the dataset
df_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 6 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Provinsi                            41 non-null     object 
 1   Population (Thousand)               39 non-null     float64
 2   Population Growth Rate              39 non-null     float64
 3   Percentage of Total Population      39 non-null     float64
 4   Population Density per sq.km (Km2)  39 non-null     Int64  
 5   Population Sex Ratio                39 non-null     float64
dtypes: Int64(1), float64(4), object(1)
memory usage: 2.1+ KB


In [28]:
# Rename field title of the dataset for easier usage
df_p = df_p.rename(columns = {
    'Provinsi': 'province',
    'Population (Thousand)': 'population_thousand',
    'Population Density per sq.km (Km2)': 'population_density'
})

# Re-sort field title of the dataset for easier usage
df_p = df_p[[
    'province', 'population_thousand', 'population_density'
]]

# Show the result
df_p.head()

,province,population_thousand,population_density
0,Aceh,5695.9,100
1,Sumatera Utara,15978.6,221
2,Sumatera Barat,5991.6,142
3,Riau,6892.4,77
4,Jambi,3811.7,78


Findings:
- All column in the hospitals dataset has no Null values.
- All column in the population dataset has Null values.
- All column in the hospitals dataset considered to be used in the future steps.
- Only several column in the population dataset will be use in the future steps.
  - `province`
  - `population`
  - `population_density`

### Detect Duplicate Values in Key Columns

The primary key in each dataset acts as the unique identifier for every row. This step checks whether duplicate values exist in the primary key fields.

In [29]:
# Check the duplicates for id field in hospital dataset
id_dup = duckdb.sql("""
SELECT COUNT(id) - COUNT(DISTINCT id)
FROM df_h
""").fetchone()

# Check the duplicates for province field in population dataset
province_dup = duckdb.sql("""
SELECT COUNT(province) - COUNT(DISTINCT province)
FROM df_p
""").fetchone()

# Show the results
print("h.id duplicates:", id_dup[0])
print("p.province duplicates:", province_dup[0])

h.id duplicates: 0
p.province duplicates: 0


Findings: Key in both datasets has no duplicate values.

### Detect Zero or Negative Numbers in Specific Columns

After reviewing the column descriptions, numerical values are inspected. The `total_beds`, `total_labors`, and `total_services` columns should not contain zero or negative values, so this step identifies any invalid records.

In [30]:
# Check zero or negative number of beds
total_beds_0 = duckdb.sql("""
SELECT count(total_beds)
FROM df_h
WHERE total_beds <= 0
""").fetchone()

# Check zero or negative number of services
total_services_0 = duckdb.sql("""
SELECT count(total_services)
FROM df_h
WHERE total_services <= 0
""").fetchone()

# Check zero or negative number of labors
total_labors_0 = duckdb.sql("""
SELECT count(total_labors)
FROM df_h
WHERE total_labors <= 0
""").fetchone()

# Show the results
print("h.total_beds with zero or negative:", total_beds_0[0])
print("h.total_services with zero or negative:", total_services_0[0])
print("h.total_labors with zero or negative:", total_labors_0[0])

h.total_beds with zero or negative: 15
h.total_services with zero or negative: 0
h.total_labors with zero or negative: 357


Findings: There are zero or negative numbers in `total_beds` and `total_labors`, while `total_services` does not have it.

### Detect Outlier Anomalies with Z-Score Method

Besides invalid values, large datasets may also contain outliers. This step uses the Z-score method to identify extreme values, while domain knowledge is used to determine whether the detected values should be treated as anomalies.

In [31]:
# Check outliers in total_beds field
z_beds_an = duckdb.sql("""
WITH cal AS (
  SELECT
    AVG(total_beds) AS avg,
    STDDEV(total_beds) AS std_dev,
FROM df_h
),
z_score_cal AS (
  SELECT
    h.id,
    h.hospital_name,
    h.city,
    h.total_beds,
    (h.total_beds - c.avg) / c.std_dev AS z_score,
  FROM df_h h
  CROSS JOIN cal c
)
SELECT *
FROM z_score_cal
WHERE z_score > 3 OR z_score < -3
ORDER BY z_score DESC
""").df()

# Show the results
display(z_beds_an.head())
display(z_beds_an.tail())

,id,hospital_name,city,total_beds,z_score
0,3310015,RS Umum Pusat Dr. Soeradji Tirtonegoro,Klaten,30343,54.827126
1,3101002,RS Umum Daerah Kepulauan Seribu,Kepulauan Seribu,1966,3.322501


,id,hospital_name,city,total_beds,z_score
0,3310015,RS Umum Pusat Dr. Soeradji Tirtonegoro,Klaten,30343,54.827126
1,3101002,RS Umum Daerah Kepulauan Seribu,Kepulauan Seribu,1966,3.322501


In [32]:
# Check outliers in total_services field
z_serv_an = duckdb.sql("""
WITH cal AS (
  SELECT
    AVG(total_services) AS avg,
    STDDEV(total_services) AS std_dev,
FROM df_h
),
z_score_cal AS (
  SELECT
    h.id,
    h.hospital_name,
    h.city,
    h.total_services,
    (h.total_services - c.avg) / c.std_dev AS z_score,
  FROM df_h h
  CROSS JOIN cal c
)
SELECT *
FROM z_score_cal
WHERE z_score > 3 OR z_score < -3
ORDER BY z_score DESC
""").df()

# Show the results
display(z_serv_an.head())
display(z_serv_an.tail())

,id,hospital_name,city,total_services,z_score
0,3310015,RS Umum Pusat Dr. Soeradji Tirtonegoro,Klaten,419,12.356265
1,3404015,RSUP Dr. Sardjito,Sleman,280,7.811389
2,3374010,RS Umum Pusat Dr. Kariadi,Kota Semarang,277,7.713299
3,3471373,RS Happy Land Medical Centre,Kota Yogyakarta,234,6.307330
4,3204075,RS Umum Mitra Kasih,Kota Cimahi,218,5.784179


,id,hospital_name,city,total_services,z_score
40,1275911,RS Umum Siloam Dhirga Surya,Kota Medan,134,3.037635
41,1471226,RS Umum Awal Bros Pekanbaru,Kota Pekanbaru,134,3.037635
42,7371408,RS Universitas Hasanuddin,Kota Makassar,134,3.037635
43,3172495,RS EMC Pulomas,Kota Jakarta Timur,133,3.004938
44,3204086,RS Umum Daerah Al Ihsan Provinsi Jawa Barat,Bandung,133,3.004938


In [33]:
# Check outliers in total_labors field
z_labo_an = duckdb.sql("""
WITH cal AS (
  SELECT
    AVG(total_labors) AS avg,
    STDDEV(total_labors) AS std_dev,
FROM df_h
),
z_score_cal AS (
  SELECT
    h.id,
    h.hospital_name,
    h.city,
    h.total_labors,
    (h.total_labors - c.avg) / c.std_dev AS z_score,
  FROM df_h h
  CROSS JOIN cal c
)
SELECT *
FROM z_score_cal
WHERE z_score > 3 OR z_score < -3
ORDER BY z_score DESC
""").df()

# Show the results
display(z_labo_an.head())
display(z_labo_an.tail())

,id,hospital_name,city,total_labors,z_score
0,3578016,RS Umum Daerah Dr. Soetomo,Kota Surabaya,7939,20.078369
1,3173014,RSUPN Dr. Cipto Mangunkusumo,Kota Jakarta Pusat,6325,15.857136
2,3374010,RS Umum Pusat Dr. Kariadi,Kota Semarang,3871,9.438979
3,3171012,RSUP Fatmawati,Kota Jakarta Selatan,3736,9.085902
4,3404015,RSUP Dr. Sardjito,Sleman,3454,8.348363


,id,hospital_name,city,total_labors,z_score
41,3277031,RS Umum Tk. II Dustira,Kota Cimahi,1422,3.033898
42,3174260,RS Anak dan Bunda Harapan Kita,Kota Jakarta Barat,1418,3.023437
43,3504012,RS Umum Daerah Dr. Iskak Tulungagung,Tulungagung,1418,3.023437
44,3215012,RS Umum Daerah Karawang,Karawang,1417,3.020821
45,3571016,RS Umum Daerah Gambiran,Kota Kediri,1415,3.015591


Findings: There are 2 value considered as anomalies in `total_beds`, while `total_services` and `total_labors` does not have it.

### Compare Shared Column between Datasets

The shared column is used as the join key between both datasets, so its values must be consistent. This step identifies any differences in the shared column before joining.

In [34]:
# Check province field differences between datasets
diff_prov = duckdb.sql("""
WITH group_prov AS (
  SELECT province
  FROM df_h
  GROUP BY province
)
SELECT
  h.province AS province_h,
  p.province AS province_p,
FROM df_p AS p
FULL JOIN group_prov AS h
  ON p.province = h.province
WHERE p.province is Null or h.province is Null
""").df()

# Show the result
display(diff_prov)

,province_h,province_p
0,Yogyakarta,None
1,None,None
2,None,Indonesia
3,None,1Proyeksi Penduduk Indonesia 2020-2050 Hasil S...
4,None,Keterangan
5,None,DI Yogyakarta


Findings: From all of item in shared column of `Province`, only `"Yogyakarta"` has different value, which has `"DI Yogyakarta"` in the other dataset.

# Data Preparation

Perform data cleaning based on the findings from the Data Understanding phase. In addition, reformat the datasets to improve consistency and simplify further analysis.

### Handling Null Values

In [35]:
# Show null values in population dataset
null_pop = duckdb.sql("""
SELECT *
FROM df_p
WHERE
  province is NULL
  OR population_thousand is NULL
  OR population_density is NULL
""").df()

# Show the result
display(null_pop)

,province,population_thousand,population_density
0,None,NaN,<NA>
1,Keterangan,NaN,<NA>
2,1Proyeksi Penduduk Indonesia 2020-2050 Hasil S...,NaN,<NA>


Since all missing values belong to unused columns, those columns can be removed from the dataset.

In [36]:
# Filter out null values in population dataset
cleaned_df_p = duckdb.sql("""
SELECT *
FROM df_p
WHERE
  province is not NULL
  AND province != 'Indonesia'
  AND population_thousand is not NULL
  AND population_density is not NULL
ORDER BY province ASC
""").df()

# Show the result
display(cleaned_df_p)

,province,population_thousand,population_density
0,Aceh,5695.9,100
1,Bali,4488.2,804
2,Banten,12641.3,1351
3,Bengkulu,2163.3,108
4,DI Yogyakarta,3802.7,1199
5,DKI Jakarta,10669.7,16129
6,Gorontalo,1256.4,104
7,Jambi,3811.7,78
8,Jawa Barat,51163.9,1381
9,Jawa Tengah,38565.0,1123


### Handling Zero or Negative Numbers

In [37]:
# Show zero or negative number in hospital dataset
beds_labors_0 = duckdb.sql("""
SELECT hospital_name, total_beds, total_labors
FROM df_h
WHERE total_beds <= 0 OR total_labors <= 0
""").df()

# Show the result
display(beds_labors_0)

,hospital_name,total_beds,total_labors
0,RS Gigi Mulut Universitas Syiah Kuala,11,0
1,RS Gayo Medical Centre,60,0
2,RS Tanoh Gayo,54,0
3,RS Umum Daerah Type D Pratama T. Cut Ali,0,0
4,RS Umum Daerah dr. Muchtar Hasbi,28,0
...,...,...,...
354,RS Pratama Elvrida Sara,0,9
355,RS Umum Daerah Ilaga,58,0
356,RS Umum Daerah Er Dabi,35,0
357,RS Tk. IV 17.07.01 Jenderal LB Moerdani,81,0


Rows containing invalid values are not removed because other columns remain valid. Instead, the invalid values are replaced with Null so they are excluded from future calculations.

In [38]:
# Convert zero or negative numbers into Null value
cleaned_df_h = duckdb.sql("""
SELECT * EXCLUDE (total_beds, total_labors),
  CASE
    WHEN total_beds <= 0 THEN Null
    ELSE total_beds
  END AS total_beds,
  CASE
    WHEN total_labors <= 0 THEN Null
    ELSE total_labors
  END AS total_labors,
FROM df_h
""").df()

# Show the result
display(cleaned_df_h)

,id,province,city,hospital_name,type,class,total_services,ownership,blu_status,address,total_beds,total_labors
0,1110053,Aceh,Kota Lhokseumawe,RS Arun Lhokseumawe,Rumah Sakit Umum,C,36,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Plaju Komplek Perumahan PT Arun Batuphat T...,218,328
1,1106014,Aceh,Aceh Tengah,RS Umum Fandika,Rumah Sakit Umum,D,15,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Terminal Simpang Wariji Blangkolak 1 Kec. ...,45,45
2,1171110,Aceh,Kota Banda Aceh,RS Umum Daerah Meuraxa,Rumah Sakit Umum,B,77,Pemkot,BLUD,Jl. Soekarno Hatta Km. 2 Desa Mibo Kecamatan B...,310,487
3,1171163,Aceh,Kota Banda Aceh,RS Gigi Mulut Universitas Syiah Kuala,Rumah Sakit Khusus Gigi dan Mulut,B,24,Kementerian Lain,BLU,Jl. Prof A. Madjid Ibrahim I No. 5 Banda Aceh ...,11,<NA>
4,1102027,Aceh,Kota Subulussalam,RS Umum Daerah Kota Subulussalam,Rumah Sakit Umum,C,34,Pemkot,BLUD,Jl. Hamzah Fansyuri (Subulussalam-Rundeng) K...,189,537
...,...,...,...,...,...,...,...,...,...,...,...,...
3150,9232011,Papua Pegunungan,Yalimo,RS Umum Daerah Er Dabi,Rumah Sakit Umum,D,18,Pemkab,Non BLU/BLUD,Jl. Trans Wamena Jayapura KM. 123 Heahobak Dis...,35,<NA>
3151,9231002,Papua Pegunungan,Mamberamo Tengah,RS Umum Daerah Lukas Enembe Kab. Memberamo Tengah,Rumah Sakit Umum,D PRATAMA,23,Pemkab,Non BLU/BLUD,"Jln. Poros Gimbis Nomor 1, Distrik Kobakma, Ka...",85,47
3152,9201046,Papua Selatan,Merauke,RS Tk. IV 17.07.01 Jenderal LB Moerdani,Rumah Sakit Umum,D,24,TNI AD,Non BLU/BLUD,"Jl. Poros L.B. Moerdani, Kel. Kamangi, Kec. Ta...",81,<NA>
3153,9212027,Papua Tengah,Mimika,RS TNI AD Tk. IV Timika,Rumah Sakit Umum,D,43,TNI AD,Non BLU/BLUD,Jl. Agimuga Mile 32 Desa Karang Senang Distrik...,60,<NA>


### Handling Outlier Anomalies

In [39]:
# Show the outliers in hospital dataset
z_beds_an = duckdb.sql("""
WITH cal AS (
  SELECT
    AVG(total_beds) AS avg,
    STDDEV(total_beds) AS std_dev,
FROM cleaned_df_h
),
z_score_cal AS (
  SELECT
    h.id,
    h.hospital_name,
    h.city,
    h.total_beds,
    (h.total_beds - c.avg) / c.std_dev AS z_score,
  FROM cleaned_df_h h
  CROSS JOIN cal c
)
SELECT *
FROM z_score_cal
WHERE z_score > 3 OR z_score < -3
ORDER BY z_score DESC
""").df()

# Show the result
display(z_beds_an)

,id,hospital_name,city,total_beds,z_score
0,3310015,RS Umum Pusat Dr. Soeradji Tirtonegoro,Klaten,30343,54.703322
1,3101002,RS Umum Daerah Kepulauan Seribu,Kepulauan Seribu,1966,3.313898


Outlier values are replaced with Null instead of removing the entire row, preserving other valid information. Based on the Z-score analysis, values greater than or equal to 1966 are treated as outliers.

In [40]:
# Convert the outliers into Null value
cleaned_df_h = duckdb.sql("""
SELECT * EXCLUDE (total_beds),
  CASE
    WHEN total_beds >= 1966 THEN Null
    ELSE total_beds
  END AS total_beds,
FROM cleaned_df_h
""").df()

# Show the result
display(cleaned_df_h)

,id,province,city,hospital_name,type,class,total_services,ownership,blu_status,address,total_labors,total_beds
0,1110053,Aceh,Kota Lhokseumawe,RS Arun Lhokseumawe,Rumah Sakit Umum,C,36,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Plaju Komplek Perumahan PT Arun Batuphat T...,328,218
1,1106014,Aceh,Aceh Tengah,RS Umum Fandika,Rumah Sakit Umum,D,15,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Terminal Simpang Wariji Blangkolak 1 Kec. ...,45,45
2,1171110,Aceh,Kota Banda Aceh,RS Umum Daerah Meuraxa,Rumah Sakit Umum,B,77,Pemkot,BLUD,Jl. Soekarno Hatta Km. 2 Desa Mibo Kecamatan B...,487,310
3,1171163,Aceh,Kota Banda Aceh,RS Gigi Mulut Universitas Syiah Kuala,Rumah Sakit Khusus Gigi dan Mulut,B,24,Kementerian Lain,BLU,Jl. Prof A. Madjid Ibrahim I No. 5 Banda Aceh ...,<NA>,11
4,1102027,Aceh,Kota Subulussalam,RS Umum Daerah Kota Subulussalam,Rumah Sakit Umum,C,34,Pemkot,BLUD,Jl. Hamzah Fansyuri (Subulussalam-Rundeng) K...,537,189
...,...,...,...,...,...,...,...,...,...,...,...,...
3150,9232011,Papua Pegunungan,Yalimo,RS Umum Daerah Er Dabi,Rumah Sakit Umum,D,18,Pemkab,Non BLU/BLUD,Jl. Trans Wamena Jayapura KM. 123 Heahobak Dis...,<NA>,35
3151,9231002,Papua Pegunungan,Mamberamo Tengah,RS Umum Daerah Lukas Enembe Kab. Memberamo Tengah,Rumah Sakit Umum,D PRATAMA,23,Pemkab,Non BLU/BLUD,"Jln. Poros Gimbis Nomor 1, Distrik Kobakma, Ka...",47,85
3152,9201046,Papua Selatan,Merauke,RS Tk. IV 17.07.01 Jenderal LB Moerdani,Rumah Sakit Umum,D,24,TNI AD,Non BLU/BLUD,"Jl. Poros L.B. Moerdani, Kel. Kamangi, Kec. Ta...",<NA>,81
3153,9212027,Papua Tengah,Mimika,RS TNI AD Tk. IV Timika,Rumah Sakit Umum,D,43,TNI AD,Non BLU/BLUD,Jl. Agimuga Mile 32 Desa Karang Senang Distrik...,<NA>,60


### Standardize Province Names

In [41]:
# Show the value that is different
diff_prov = duckdb.sql("""
WITH group_prov AS (
  SELECT province
  FROM cleaned_df_h
  GROUP BY province
)
SELECT
  h.province AS province_h,
  p.province AS province_p,
FROM cleaned_df_p AS p
FULL JOIN group_prov AS h
  ON p.province = h.province
WHERE p.province is Null or h.province is Null
""").df()

# Show the result
display(diff_prov)

,province_h,province_p
0,Yogyakarta,None
1,None,DI Yogyakarta


The only inconsistent province name is Yogyakarta. Therefore, the province name in the population dataset is standardized before joining.

In [42]:
# Convert the different value so it will be same
cleaned_df_p = duckdb.sql("""
SELECT * EXCLUDE (province),
  CASE
    WHEN province = 'DI Yogyakarta' THEN 'Yogyakarta'
    ELSE province
  END AS province,
FROM cleaned_df_p
""").df()

# Show the result
display(cleaned_df_p)

,population_thousand,population_density,province
0,5695.9,100,Aceh
1,4488.2,804,Bali
2,12641.3,1351,Banten
3,2163.3,108,Bengkulu
4,3802.7,1199,Yogyakarta
5,10669.7,16129,DKI Jakarta
6,1256.4,104,Gorontalo
7,3811.7,78,Jambi
8,51163.9,1381,Jawa Barat
9,38565.0,1123,Jawa Tengah


After both datasets have been cleaned, they are joined into a single dataset for subsequent analysis.

In [43]:
# Join both cleaned hospital and population dataset
joined = duckdb.sql("""
SELECT *
FROM cleaned_df_h
INNER JOIN cleaned_df_p
  USING (province)
ORDER BY
  province ASC,
  city ASC,
  type ASC,
  class ASC
""").df()

# Show the result
display(joined)

,id,province,city,hospital_name,type,class,total_services,ownership,blu_status,address,total_labors,total_beds,population_thousand,population_density
0,1107014,Aceh,Aceh Barat,RS Umum Daerah Cut Nyak Dhien,Rumah Sakit Umum,B,16,Pemkab,BLUD,Jl. Gajah Mada Meulaboh,812,267,5695.9,100
1,1107016,Aceh,Aceh Barat,RS Umum Harapan Sehat,Rumah Sakit Umum,D,25,SWASTA/LAINNYA,Non BLU/BLUD,"Jl. Sisingamangaraja, Dusun III, Desa Gampa, K...",53,51,5695.9,100
2,1107017,Aceh,Aceh Barat,RS Tk. IV IM 07.02,Rumah Sakit Umum,D,23,TNI AD,Non BLU/BLUD,Jl. Pocut Baren Ujung Karang Desa Suak Indrapu...,20,56,5695.9,100
3,1107015,Aceh,Aceh Barat,RS Umum Montella,Rumah Sakit Umum,D,16,SWASTA/LAINNYA,Non BLU/BLUD,"Jl. Beringin jaya , kel. seunubok Kec. Johan...",55,51,5695.9,100
4,1112011,Aceh,Aceh Barat Daya,RS Umum Daerah Teungku Peukan,Rumah Sakit Umum,C,96,Pemkab,BLUD,Jl. Nasional Padang Meurante Desa Ujung Padang...,1100,256,5695.9,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3150,3404190,Yogyakarta,Sleman,RS Umum Mitra Sehat,Rumah Sakit Umum,D,22,Perusahaan,Non BLU/BLUD,"Jl. Wates KM 9 Balecatur, Gamping",11,50,3802.7,1199
3151,3404196,Yogyakarta,Sleman,RS Umum Bunga Bangsa Medika,Rumah Sakit Umum,D,18,SWASTA/LAINNYA,Non BLU/BLUD,Jl. Lili Kembang Nomor 17 Maguwoharjo Depok,8,50,3802.7,1199
3152,3404033,Yogyakarta,Sleman,Charitas Hospital Klepu,Rumah Sakit Umum,D,22,Organisasi Sosial,Non BLU/BLUD,Klepu Sendangmulyo Minggir,150,49,3802.7,1199
3153,3404081,Yogyakarta,Sleman,RS Umum Condong Catur,Rumah Sakit Umum,D,50,SWASTA/LAINNYA,Non BLU/BLUD,"Jl Manggis No.6, Gempol, Condong Catur, Depok,...",219,80,3802.7,1199


### Merge Datasets

In [46]:
# Re-sort the field for easier usage in future steps
joined = joined[[
    'id', 'province', 'population_thousand', 'population_density',
    'hospital_name', 'type', 'class', 'total_beds', 'total_services',
    'total_labors', 'ownership', 'blu_status', 'city', 'address'
]]

# Show the result
joined.head()

,id,province,population_thousand,population_density,hospital_name,type,class,total_beds,total_services,total_labors,ownership,blu_status,city,address
0,1107014,Aceh,5695.9,100,RS Umum Daerah Cut Nyak Dhien,Rumah Sakit Umum,B,267,16,812,Pemkab,BLUD,Aceh Barat,Jl. Gajah Mada Meulaboh
1,1107016,Aceh,5695.9,100,RS Umum Harapan Sehat,Rumah Sakit Umum,D,51,25,53,SWASTA/LAINNYA,Non BLU/BLUD,Aceh Barat,"Jl. Sisingamangaraja, Dusun III, Desa Gampa, K..."
2,1107017,Aceh,5695.9,100,RS Tk. IV IM 07.02,Rumah Sakit Umum,D,56,23,20,TNI AD,Non BLU/BLUD,Aceh Barat,Jl. Pocut Baren Ujung Karang Desa Suak Indrapu...
3,1107015,Aceh,5695.9,100,RS Umum Montella,Rumah Sakit Umum,D,51,16,55,SWASTA/LAINNYA,Non BLU/BLUD,Aceh Barat,"Jl. Beringin jaya , kel. seunubok Kec. Johan..."
4,1112011,Aceh,5695.9,100,RS Umum Daerah Teungku Peukan,Rumah Sakit Umum,C,256,96,1100,Pemkab,BLUD,Aceh Barat Daya,Jl. Nasional Padang Meurante Desa Ujung Padang...


In [47]:
# Export the clean joined dataset for further analysis
joined.to_csv('hospital_population_cleaned.csv', index=False)